In [4]:
import numpy as np
import math
from collections import deque

def compute_n_small_roots(M, n, max_iter=1000):
    r = len(M)
    
    # 构建双线性型
    B = np.zeros((r, r))
    for i in range(r):
        B[i, i] = 1.0
        for j in range(i+1, r):
            m = M[i][j]
            if m == float('inf'):
                B[i, j] = -1.0
                B[j, i] = -1.0
            else:
                B[i, j] = -math.cos(math.pi / m)
                B[j, i] = B[i, j]
    
    # 初始化
    roots = {}
    for i in range(r):
        coeffs = tuple(1 if j == i else 0 for j in range(r))
        roots[coeffs] = 0
    
    queue = deque(roots.keys())
    iteration = 0
    
    while queue and iteration < max_iter:
        iteration += 1
        coeffs = queue.popleft()
        depth = roots[coeffs]
        beta = np.array(coeffs)
        
        for i in range(r):
            alpha = np.zeros(r)
            alpha[i] = 1
            
            B_ab = np.dot(alpha, np.dot(B, beta))
            new_beta = beta - 2 * B_ab * alpha
            new_coeffs = tuple(np.round(new_beta, decimals=8))
            
            if not all(c >= -1e-8 for c in new_coeffs):
                continue
            if all(abs(c) < 1e-8 for c in new_coeffs):
                continue
            
            new_depth = depth + 1 if B_ab <= -1 else depth  #############################
            
            if new_coeffs not in roots:
                roots[new_coeffs] = new_depth
                queue.append(new_coeffs)
            elif new_depth < roots[new_coeffs]:
                roots[new_coeffs] = new_depth
                queue.append(new_coeffs)
    
    unique_roots = []
    for coeffs, depth in roots.items():
        found = False
        for existing, _ in unique_roots:
            if len(existing) == len(coeffs) and all(abs(existing[j] - coeffs[j]) < 1e-6 for j in range(len(coeffs))):
                found = True
                break
        if not found:
            unique_roots.append((coeffs, depth))
    
    result = [(c, d) for c, d in unique_roots if d <= n]
    result.sort(key=lambda x: (x[1], sum(x[0])))
    return result


def print_result(M, n):
    r = len(M)
    result = compute_n_small_roots(M, n)

    print(f"\n找到 {len(result)} 个 {n}-small roots:")
    print("-" * 80)
    
    header = "  " + "  ".join([f"c{i+1:>10}" for i in range(r)]) + "  | depth"
    print(header)
    print("-" * 80)
    
    for coeffs, depth in result:
        row = "  ".join([f"{c:10.6f}" for c in coeffs])
        print(f"  {row}  |  {depth:2d}")
    
    print("-" * 80)
    
    by_depth = {}
    for _, depth in result:
        by_depth[depth] = by_depth.get(depth, 0) + 1
    
    print("\n按深度统计:")
    for depth in sorted(by_depth.keys()):
        print(f"  深度 {depth}: {by_depth[depth]} 个根")


if __name__ == "__main__":
    M = []
    while True:
        line = input().strip()
        if line.lower() == 'done':
            break
        if line:
            row = []
            for val in line.split():
                if val.lower() == 'inf':
                    row.append(float('inf'))
                else:
                    row.append(float(val))
            M.append(row)
    
    if not M:
        print("未输入矩阵，程序退出")
        exit()
    
    n = int(input("\n请输入 n: "))
    
    print_result(M, n)

 1 3 4
 3 1 5
 4 5 1
 done

请输入 n:  8



找到 597 个 8-small roots:
--------------------------------------------------------------------------------
  c         1  c         2  c         3  | depth
--------------------------------------------------------------------------------
    1.000000    0.000000    0.000000  |   0
    0.000000    1.000000    0.000000  |   0
    0.000000    0.000000    1.000000  |   0
    1.000000    1.000000    0.000000  |   0
    1.000000    0.000000    1.414214  |   0
    1.414214    0.000000    1.000000  |   0
    0.000000    1.000000    1.618034  |   0
    0.000000    1.618034    1.000000  |   0
    0.000000    1.618034    1.618034  |   0
    1.000000    1.000000    3.032248  |   1
    1.414214    3.032248    1.000000  |   1
    3.032248    1.618034    1.000000  |   1
    1.000000    3.288246    1.414214  |   1
    3.288246    1.000000    1.618034  |   1
    3.032248    3.032248    1.000000  |   1
    3.906280    1.618034    1.618034  |   1
    4.288246    1.000000    3.032248  |   2
    1.000000    